# Test Gemini Multimodal — Gambar & Video dari URL Berita

Notebook ini untuk menjawab pertanyaan: **apakah free tier Gemini API benar-benar bisa memproses gambar dan video langsung** (bukan lewat OCR/Whisper dulu)?

Alurnya:
1. Ambil link video/gambar dari halaman berita pakai `newspaper3k`
2. Kirim gambar utama halaman langsung ke Gemini (inline bytes)
3. Kirim video langsung ke Gemini — dicoba 2 cara: (a) link YouTube langsung, (b) download dulu lalu upload via Files API
4. Semua error ditampilkan apa adanya (tidak disembunyikan) supaya kelihatan persis kenapa gagal kalau gagal

**Sebelum run:** siapkan API key asli dari [aistudio.google.com/apikey](https://aistudio.google.com/apikey) — pastikan diawali `AIzaSy`. Key akan diminta lewat kotak input tersembunyi (`getpass`), **jangan** ditulis langsung di kode supaya tidak ikut tersimpan kalau notebook ini dibagikan.

## 1. Install dependency

In [1]:
!pip install -q newspaper3k lxml_html_clean google-genai yt-dlp requests
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 62.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 8.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## 2. Masukkan API key & sanity check (teks biasa)

Ini dulu yang dites — kalau ini saja gagal, masalahnya ada di key-nya, bukan di kemampuan multimodal.

In [2]:
from getpass import getpass
from google import genai

GEMINI_API_KEY = getpass("Masukkan Gemini API key (diawali AIzaSy): ")
MODEL = "gemini-3.6-flash"  # ganti kalau Google sudah rilis versi lebih baru

client = genai.Client(api_key=GEMINI_API_KEY)

try:
    resp = client.models.generate_content(model=MODEL, contents="Balas dengan kata OK saja.")
    print("SANITY CHECK TEKS: SUKSES ->", resp.text)
except Exception as e:
    print("SANITY CHECK TEKS: GAGAL ->", type(e).__name__, str(e)[:500])

Masukkan Gemini API key (diawali AIzaSy): ··········
SANITY CHECK TEKS: SUKSES -> OK


## 3. Ekstrak gambar & video dari URL pakai newspaper3k

In [3]:
from newspaper import Article

url = "https://www.kompas.tv/nasional/691363/febrie-adriansyah-dan-bukti-kasusnya-dilimpahkan-ke-penuntut-umum?medium=headline&so=1"  # <-- ganti dengan URL asli, idealnya yang punya gambar & video

article = Article(url, language='id')
article.download()
article.parse()

print("Judul       :", article.title)
print("Top image   :", article.top_image)
print("Jumlah gambar:", len(article.images))
print("Video/movies:", article.movies)

Judul       : Febrie Adriansyah dan Bukti Kasusnya Dilimpahkan ke Penuntut Umum
Top image   : https://media-origin.kompas.tv/library/image/content_article/article_img/20260918045744.webp
Jumlah gambar: 5
Video/movies: []


## 4. Test 1 — Kirim gambar langsung ke Gemini (inline bytes)

Kalau `article.top_image` kosong, isi manual variabel `image_url` di bawah dengan link gambar lain.

In [4]:
import requests
from google.genai import types

image_url = article.top_image or "PASTE_LINK_GAMBAR_MANUAL_DISINI"
print("Gambar yang dites:", image_url)

try:
    img_resp = requests.get(image_url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
    img_bytes = img_resp.content
    mime = img_resp.headers.get("Content-Type", "image/jpeg").split(";")[0]
    print(f"Berhasil download gambar: {len(img_bytes)} bytes, mime={mime}")

    resp = client.models.generate_content(
        model=MODEL,
        contents=[
            types.Part.from_bytes(data=img_bytes, mime_type=mime),
            "Deskripsikan isi gambar ini dalam 2-3 kalimat Bahasa Indonesia. "
            "Apakah ada tanda-tanda gambar ini editan/manipulasi? Apakah ini foto kejadian nyata atau ilustrasi/meme?",
        ],
    )
    print("\nTEST GAMBAR: SUKSES")
    print(resp.text)
except Exception as e:
    print("\nTEST GAMBAR: GAGAL ->", type(e).__name__)
    print(str(e)[:800])

Gambar yang dites: https://media-origin.kompas.tv/library/image/content_article/article_img/20260918045744.webp
Berhasil download gambar: 50974 bytes, mime=image/webp

TEST GAMBAR: SUKSES
Berikut adalah deskripsi dan analisis gambar tersebut:

**Deskripsi Gambar:**
Gambar ini menampilkan seorang pria berkacamata yang mengenakan kemeja batik biru di bagian depan dengan ekspresi wajah serius. Di bagian latar belakang, tampak seorang pria lain yang mengenakan rompi tahanan berwarna merah muda dengan angka "22", yang identik dengan rompi tahanan korupsi Kejaksaan Agung RI. Foto ini umumnya digunakan sebagai gambar sampul (*thumbnail*) atau ilustrasi berita terkait penanganan kasus hukum di Indonesia.

**Apakah ada tanda-tanda editan/manipulasi?**
**Ya, gambar ini merupakan hasil editan (kolase/penggabungan foto).** Tanda-tandanya terlihat dari batas tepi tubuh pria berkemeja batik yang dibuat agak kabur (*soft edge/blend*) agar menyatu dengan latar belakang, serta adanya perbedaan pencahay

## 5. Test 2 — Kirim video ke Gemini via link youtube

Ini fitur yang menurut dokumentasi Google masih **preview** dan pernah gagal butuh OAuth waktu saya tes. Kita cek lagi di sini apakah key kamu bisa.

In [7]:
video_url = None
for m in article.movies:
    if "youtube.com" in m or "youtu.be" in m:
        video_url = m
        break

if not video_url:
    print("Tidak ada link YouTube ditemukan di halaman ini. Isi manual kalau mau tetap tes:")
    video_url = "https://youtu.be/68aiC3V-zN8?si=LCoTVbAowyBxhLeu"

print("Video yang dites:", video_url)

try:
    resp = client.models.generate_content(
        model=MODEL,
        contents=types.Content(
            parts=[
                types.Part(file_data=types.FileData(file_uri=video_url)),
                types.Part(text="Ringkas video ini dalam 2-3 kalimat Bahasa Indonesia."),
            ]
        ),
    )
    print("\nTEST VIDEO (cara A - link langsung): SUKSES")
    print(resp.text)
except Exception as e:
    print("\nTEST VIDEO (cara A - link langsung): GAGAL ->", type(e).__name__)
    print(str(e)[:800])

Tidak ada link YouTube ditemukan di halaman ini. Isi manual kalau mau tetap tes:
Video yang dites (cara A): https://youtu.be/68aiC3V-zN8?si=LCoTVbAowyBxhLeu

TEST VIDEO (cara A - link langsung): SUKSES
Berikut adalah ringkasan video tersebut:

Video ini menjelaskan tentang neutrino atau "partikel hantu" yang hampir tidak bermassa dan sangat sulit dideteksi, serta peran Observatorium IceCube di Antarktika dalam menangkap neutrino berenergi tinggi dari luar angkasa. Observatorium raksasa ini memanfaatkan lapisan es murni di kedalaman Antarktika untuk mendeteksi jejak cahaya yang dihasilkan saat neutrino menabrak inti atom. Penemuan ini membuka era baru astronomi *multi-messenger*, memungkinkan ilmuwan mempelajari objek dan fenomena alam semesta yang tidak dapat teramati hanya dengan cahaya atau gelombang elektromagnetik.
